## The steps for creating the depth images are as follows:
1. Generating example of depth, RGB and opacity images.
2. Create depth images for each of train, val and test datasets.
3. Move the depth images to the relevant folders.
4. Verify that the process was done correctly.




***First, initialization of the dataset.***

In [ ]:
from google.colab import drive

# clearing drive cache:
!rm -rf ~/.cache/Google

drive.mount('/content/drive', force_remount=True)

In [ ]:
%cd /content/splatter-image

/content/splatter-image


In [ ]:
!ls '/content/drive/MyDrive/SRN_dataset/srn_cars' #the path of SHAPENET_DATASET_ROOT in datasets/srn.py all the other are init with empty string

cars_test  cars_train  cars_val


# Step 1: Generating example of depth, rgb and opacity images

In [ ]:
#add to scene/guassian_predictor.py at the end of the file to
#plotting and saving rgb, opacity and depth as images below


  import matplotlib.pyplot as plt # Assuming 'depth' is the tensor with shape [1, 1, 128, 128]
  depth_map = depth[0, 0].detach().cpu().numpy() # Take the first batch and first channel
  # Normalize the depth map to 0-1 range for visualization
  depth_map_normalized = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min()) # Plot the depth map using matplotlib
  plt.imshow(depth_map_normalized, cmap='plasma') # Using 'plasma' or 'gray' colormap plt.colorbar(label='Depth')
  plt.title('Depth Map')
  plt.imsave("depth_map.png", depth_map_normalized, cmap='plasma')
  print("Depth map saved as depth_map.png")

  opacity = opacity.cpu().numpy().reshape(128, 128)
  plt.title("Opacity")
  plt.imshow(opacity, cmap='gray')
  plt.colorbar()
  plt.imsave("opacity.png", opacity)

  print("opacity map saved as opacity.png")

  features_rest = features_rest.cpu().numpy().reshape(128, 128, 3, 3)  # Shape [128, 128, 3, 3]
  for i in range(3):
    features_image = features_rest[..., i, :]

    # Normalize the data to [0, 1] range if necessary
    features_image = (features_image - features_image.min()) / (features_image.max() - features_image.min())

  plt.figure(figsize=(6, 6))
  plt.title(f"Features Rest Set {i+1}")
  plt.imshow(features_image)  # Shape [128, 128, 3] - treated as RGB
  plt.title("RGB")
  plt.colorbar()
  plt.imsave("RGB.png", features_image)

  print("RGB map saved as RGB.png")

  xyz = out_dict["xyz"].cpu().numpy().reshape(128, 128, 3)  # Shape [128, 128, 3]

  # Create a combined figure with three subplots for X, Y, Z components
  plt.figure(figsize=(15, 5))

  # X Component
  plt.subplot(1, 3, 1)
  plt.title("X Component")
  plt.imshow(xyz[..., 0], cmap='coolwarm')  # Choose a colormap that enhances contrast
  plt.colorbar()

  # Y Component
  plt.subplot(1, 3, 2)
  plt.title("Y Component")
  plt.imshow(xyz[..., 1], cmap='coolwarm')
  plt.colorbar()

  # Z Component
  plt.subplot(1, 3, 3)
  plt.title("Z Component")
  plt.imshow(xyz[..., 2], cmap='coolwarm')
  plt.colorbar()

  # Display the figure with all three components
  plt.tight_layout()  # Adjust layout for better spacing
  print("XYZ map saved as xyz.png")
  plt.savefig("xyz_components.png")

  return out_dict


# Step 2: save depth images to drive.

In [ ]:
# to save the depth maps we need to follow these steps:
'''
1. to make things easy, eval.py (that use the paper model) is using the cars_test dataset. we want the depth maps of the cars_train dataset so we change the names of this dataset to be cars_test.
2. now, as we know the original size of the cars_train is 50 rgb images and the size of cars_test is 250 images so we need to fit our code to use only 50 images:
  a. in datasets/srn.py in line 56 change the test_input_idxs to be [49] instead of [64]
  b. also, in the same file, in line 140 change to range to be 50 instead of 251
  c. we want the depth map of each rgb image that we have in the train (150 in total)
    but the input_images in the model_cfg.data.input_images is 1. so change manually this
    number to be 50 at first in line 72 in eval.py and make it back to be 1 in line 101.
  d. write the next cell in scene/guassian_predictor.py under the class GaussianSplatPredictor
    and make this call in the forwad func of this class:
        save_dir = '/content/drive/MyDrive/CV_lab/grey_depth_training_images'  # Specify the path to Google Drive
        batch_size = depth.shape[0]  # Get the batch size from the depth tensor
        self.save_depth_images_to_drive(depth, batch_size, save_dir)
'''

In [ ]:

def save_depth_images_to_drive(self, depth, batch_size, save_dir):
    import matplotlib.pyplot as plt
    import os
    # Create the directory if it doesn't exist
    os.makedirs(save_dir, exist_ok=True)

    # Loop through each image in the batch
    for i in range(batch_size):
        depth_map = depth[i, 0].detach().cpu().numpy()  # Take the first channel of each image

        # Normalize the depth map to 0-1 range for visualization
        depth_map_normalized = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min())

        # Construct the filename
        filename = os.path.join(save_dir, f"depth_map_{self.example_index}.png")
        self.example_index += 1
        # Save the depth map using matplotlib
        plt.imsave(filename, depth_map_normalized, cmap='plasma')
        print(f"Depth map saved as {filename}")

#********* self.example_index is a field in the class that init to 0.




# Step 3: move depth images into folders


In [ ]:
import os
import shutil

def into_folder_50(image_dir, output_dir):
  # Create the output directory if it doesn't exist
  if not os.path.exists(output_dir):
      os.makedirs(output_dir)

  # List of image file names
  image_files = sorted(os.listdir(image_dir), key=lambda x: int(x.split('_')[2].split('.')[0]))

  # Total number of images and number of images per folder
  images_per_folder = 50
  total_images = len(image_files)
  total_folders = total_images // images_per_folder

  # Loop over the images and copy them into corresponding folders
  for i, image in enumerate(image_files):
      # Determine which folder the image belongs to (1-based indexing for folder names)
      folder_index = i // images_per_folder + 1

      # Create the folder if it doesn't exist
      folder_name = f'depth_{folder_index}'
      folder_path = os.path.join(output_dir, folder_name)

      if not os.path.exists(folder_path):
          os.makedirs(folder_path)

      # Copy the image to the folder
      source_path = os.path.join(image_dir, image)
      destination_path = os.path.join(folder_path, image)

      shutil.copy(source_path, destination_path)
      print(f"finish with {i}")

image_dir = "/content/drive/MyDrive/CV_lab/gray_depth_training_images"
output_dir = "/content/drive/MyDrive/CV_lab/depth_train"

into_folder_50(image_dir, output_dir)

In [ ]:
#rename images to be 0-49
import os

root_dir = "/content/drive/MyDrive/CV_lab/depth_train"

for sub_dir_name in os.listdir(root_dir):
    image_dir = os.path.join(root_dir, sub_dir_name)

    # Path to the folder containing the images
    #image_dir = 'path_to_your_images'  # Replace with the actual image directory path

    # List of image file names (assuming .jpg extension, adjust if necessary)
    image_files = sorted(os.listdir(image_dir), key=lambda x: int(x.split('_')[2].split('.')[0]))

    # Loop over all images in the directory
    for image in image_files:
        # Extract the numeric part of the file name (e.g., 'depth_map_123' -> 123)
        image_index = int(image.split('_')[2].split('.')[0])

        # Calculate the new index using modulo 50
        new_index = image_index % 50  # This will map the image index to the range [0, 49]

        # Create the new file name with zero-padding to 6 digits
        new_name = f"{new_index:06d}.jpg"  # Adjust extension if necessary (e.g., .png)

        # Full paths for renaming
        old_file_path = os.path.join(image_dir, image)
        new_file_path = os.path.join(image_dir, new_name)

        # Rename the file
        os.rename(old_file_path, new_file_path)
    print(f"finish with {sub_dir_name}")

print("Renaming completed.")

In [ ]:
#rename shift to the left 1->0 2->1 ... 49->48 0->49
import os

root_dir = "/content/drive/MyDrive/CV_lab/depth_train"

for sub_dir_name in os.listdir(root_dir):
    image_dir = os.path.join(root_dir, sub_dir_name)
    # Path to the folder containing the images

    # List of image file names (assuming they are numbered from 0 to 49, and with some extension like .jpg or .png)
    image_files = sorted(os.listdir(image_dir), key=lambda x: int(os.path.splitext(x)[0]))

    # Total number of images
    n = 50

    # Dictionary to map the original name to the new name
    rename_map = {}

    for i in range(n):
        original_name = os.path.join(image_dir, f"{i:06d}.jpg")  # assuming .jpg extension
        new_name = os.path.join(image_dir, f"{(i - 1) % n:06d}.jpg")  # New name follows the (i-1)%n rule
        rename_map[original_name] = new_name

    # Temporary renaming to avoid overwriting
    for original, new in rename_map.items():
        temp_name = original + "_temp"
        os.rename(original, temp_name)

    # Final renaming to the new names
    for temp_name, new in rename_map.items():
        os.rename(temp_name + "_temp", new)

    print(f"finish with {sub_dir_name}")

print("Renaming completed.")

In [ ]:
#move to depth_i folder in srn_cars
import os
import shutil

# Paths to the source and target directories
source_dir = '/content/drive/MyDrive/CV_lab/depth_train'  # Replace with the actual source directory path
target_parent_dir = '/content/drive/MyDrive/SRN_dataset/srn_cars/cars_train'  # Replace with the actual target directory path

# Function to extract the numeric part from directory names like "depth_1"
def extract_number(directory_name):
    return int(directory_name.split('_')[1])  # Extract the number after "depth_"

# Get a sorted list of source directories based on the numeric part of the name (e.g., depth_1, depth_2, ...)
source_dirs = sorted([d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))], key=extract_number)

# Get a sorted list of target subdirectories (e.g., a, b, c, ...)
target_dirs = sorted([d for d in os.listdir(target_parent_dir) if os.path.isdir(os.path.join(target_parent_dir, d))])

# Ensure that the number of source directories is less than or equal to the number of target directories
if len(source_dirs) != len(target_dirs):
    raise ValueError("There are more source directories than target subdirectories.")

# Copy each source directory into its corresponding target subdirectory
for src_dir, tgt_subdir in zip(source_dirs, target_dirs):
    # Full path to the source directory
    src_path = os.path.join(source_dir, src_dir)

    # Full path to the target subdirectory
    tgt_subdir_path = os.path.join(target_parent_dir, tgt_subdir)

    # Define the path where the source directory will be copied into the target subdirectory
    tgt_path = os.path.join(tgt_subdir_path, src_dir)  # Copy depth_1 into 'a', depth_2 into 'b', etc.

    # Copy the entire directory (recursively)
    shutil.copytree(src_path, tgt_path)

    print(f"Finished with {src_dir}")

print("Directories successfully copied into target subfolders.")

In [ ]:
#remove the _i of each folders' name
import os

# Path to the parent directory containing subfolders (e.g., "abc")
parent_dir = '/content/drive/MyDrive/SRN_dataset/srn_cars/cars_train'  # Replace with the actual path to the "abc" directory

# Loop through each subfolder (e.g., "a", "b", "c", etc.) inside "abc"
for subdir in os.listdir(parent_dir):
    subdir_path = os.path.join(parent_dir, subdir)

    # Check if it's a directory
    if os.path.isdir(subdir_path):
        # Loop through each directory inside the subfolder
        for folder in os.listdir(subdir_path):
            folder_path = os.path.join(subdir_path, folder)

            # If the folder name starts with "depth_" and it's a directory
            if os.path.isdir(folder_path) and folder.startswith("depth_"):
                # New folder name (remove "_i" and rename it to just "depth")
                new_folder_name = "depth"
                new_folder_path = os.path.join(subdir_path, new_folder_name)

                # Rename the folder
                os.rename(folder_path, new_folder_path)
    print(f"finish with {subdir}")

print("Renaming completed.")

In [ ]:
# ONLY if necessary rename the image name to be 6 chars
import os

# Path to the parent directory containing subfolders (e.g., "abc")
parent_dir = '/content/drive/MyDrive/SRN_dataset/srn_cars/cars_test'  # Replace with the actual path to the "abc" directory

# Loop through each subfolder (e.g., "a", "b", "c", etc.) inside "abc"
for subdir in os.listdir(parent_dir):
    subdir_path = os.path.join(parent_dir, subdir)

    # Check if it's a directory
    if os.path.isdir(subdir_path):
        # Construct the path to the "depth" folder inside the subdirectory
        depth_folder_path = os.path.join(subdir_path, 'depth')

        # Check if the "depth" folder exists
        if os.path.isdir(depth_folder_path):
            # Loop through each image file in the "depth" folder
            for image_name in os.listdir(depth_folder_path):
                # Check if the item is a file and has a valid image extension (e.g., .jpg, .png)
                if os.path.isfile(os.path.join(depth_folder_path, image_name)):
                    # Extract the numeric part of the image name (e.g., "0" from "0.jpg")
                    image_index = int(os.path.splitext(image_name)[0])  # Assumes name is like "0.jpg"

                    # Create the new name with zero padding to 6 digits
                    new_name = f"{image_index:06d}{os.path.splitext(image_name)[1]}"  # Preserves the original extension

                    # Full paths for renaming
                    old_image_path = os.path.join(depth_folder_path, image_name)
                    new_image_path = os.path.join(depth_folder_path, new_name)

                    # Rename the image
                    os.rename(old_image_path, new_image_path)

print("Image renaming completed.")


# Step 4: combine folders, for this step, run for test and val folders

In [ ]:
# combine folders and print outputs

import os
import shutil

root = "/content/drive/MyDrive/SRN_dataset/srn_cars/cars_test"
c = 0
for folder in os.listdir(root):
  rgb1 = os.path.join(root, folder)
  rgb = os.path.join(rgb1, "rgb")
  if len(os.listdir(rgb)) == 0:
    print(rgb, end = " ")
    print(len(os.listdir(rgb)))
    #print("deleting folder: ", rgb1)
    shutil.rmtree(rgb1)

    c+=1
  #break
print(c)
print(len(os.listdir(root)))

In [ ]:
#rename the each folder of images to be instead of 0-49 to be 0-249. the first relevant folder be 0-49 the second will be 50-99 etc.
#doing it on the rgb, pose and depth folder in val and test
import os

root_dir = "/content/drive/MyDrive/SRN_dataset/srn_cars/cars_test"


for sub_dir_name in os.listdir(root_dir):
    image_dir = os.path.join(root_dir, sub_dir_name)
    image_files = sorted(os.listdir(image_dir))
    index_dir = image_dir[-1]
    print(index_dir)

#     # Path to the folder containing the images
      # Replace with the actual image directory path

    # List of image file names (assuming .jpg extension, adjust if necessary)
    rgb_dir = os.path.join(image_dir, "rgb")
    pose_dir = os.path.join(image_dir, "pose")
    depth_dir = os.path.join(image_dir, "depth")

    rgb_files = sorted(os.listdir(rgb_dir))
    pose_files = sorted(os.listdir(pose_dir))
    depth_files = sorted(os.listdir(depth_dir))

    rgb_files = sorted(os.listdir(rgb_dir), key=lambda x: int(x.split('.')[0]))
    pose_files = sorted(os.listdir(pose_dir), key=lambda x: int(x.split('.')[0]))
    depth_files = sorted(os.listdir(depth_dir), key=lambda x: int(x.split('.')[0]))

    # Loop over all images in the directory
    for depth in depth_files:
        # Extract the numeric part of the file name (e.g., 'depth_map_123' -> 123)
        image_index = int(depth.split('.')[0])

        # Calculate the new index using modulo 50
        new_index = image_index + (50*(int(index_dir)-1))  # This will map the image index to the range [0, 49]

        # Create the new file name with zero-padding to 6 digits
        new_name = f"{new_index:06d}.png"  # Adjust extension if necessary (e.g., .png)

        # Full paths for renaming
        old_file_path = os.path.join(depth_dir, depth)
        new_file_path = os.path.join(depth_dir, new_name)
        #print (f"from {depth} to {new_name}")

        # Rename the file
        os.rename(old_file_path, new_file_path)
    print(f"finish with {sub_dir_name}")

# print("Renaming completed.")

In [ ]:
#merge 5 folder to 1 folder. relevant for rgb, pose and depth
import os
import shutil
from collections import defaultdict


# Function to extract the common part of folder names
def extract_common_prefix(folder_name):
    # Find the first occurrence of a digit at the end of the folder name
    #print(re.sub(r'\d+$', '', folder_name), end=" ")
    return folder_name[:-1]

# Function to find and move folders with common prefixes
def find_and_move_folders(base_path):
    # Dictionary to group folders by their common prefix
    grouped_folders = defaultdict(list)

    # Iterate through all folders in the base path
    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)
        if os.path.isdir(folder_path):
            # Extract the common prefix (anything before the final digits)
            common_prefix = extract_common_prefix(folder)
            #print(common_prefix, end=" ")
            grouped_folders[common_prefix].append(folder_path)
            #print(grouped_folders)

    # Now move and merge each group of folders
    for common_prefix, folder_list in grouped_folders.items():
        target_folder = os.path.join(base_path, common_prefix)
        move_folders(folder_list, target_folder)

# Function to move and merge directories
def move_folders(root_folders, target_folder):
    # Create target folder structure
    rgb_target = os.path.join(target_folder, "rgb")
    depth_target = os.path.join(target_folder, "depth")
    pose_target = os.path.join(target_folder, "pose")

    # Create the target folders if they don't exist
    os.makedirs(rgb_target, exist_ok=True)
    os.makedirs(depth_target, exist_ok=True)
    os.makedirs(pose_target, exist_ok=True)

    intrinsics_file_moved = False

    # Iterate over the list of root folders
    for folder in root_folders:
        rgb_source = os.path.join(folder, "rgb")
        depth_source = os.path.join(folder, "depth")
        pose_source = os.path.join(folder, "pose")
        intrinsics_file = os.path.join(folder, "intrinsics.txt")

        # Move RGB images
        for file_name in os.listdir(rgb_source):
            full_file_name = os.path.join(rgb_source, file_name)
            if os.path.isfile(full_file_name):
                shutil.move(full_file_name, rgb_target)

        # Move Depth images
        for file_name in os.listdir(depth_source):
            full_file_name = os.path.join(depth_source, file_name)
            if os.path.isfile(full_file_name):
                shutil.move(full_file_name, depth_target)

        # Move Pose txt files
        for file_name in os.listdir(pose_source):
            full_file_name = os.path.join(pose_source, file_name)
            if os.path.isfile(full_file_name):
                shutil.move(full_file_name, pose_target)

        # Move intrinsics.txt only once
        if not intrinsics_file_moved and os.path.isfile(intrinsics_file):
            shutil.move(intrinsics_file, target_folder)
            intrinsics_file_moved = True

        # Remove the now-empty folder
        shutil.rmtree(folder)

    print(f"finish with {root_folders}")

# Specify the base path where all the folders are located
base_path = "/content/drive/MyDrive/SRN_dataset/srn_cars/cars_test"

# Call the function to find and move folders
find_and_move_folders(base_path)

print("Folders have been moved and merged successfully!")

# Step 5: Verifying correctness of the created images and folders

In [ ]:


#script to say where there are duplicate images .png or .txt files from train folder of pose and rgb folders correspondingly

# Import necessary libraries
import os
from google.colab import drive

# Define the path for the cars_train folder
cars_train_path = ('/content/drive/MyDrive/SRN_dataset/srn_cars/cars_train',
                  '/content/drive/MyDrive/SRN_dataset/srn_cars/cars_val',
                  '/content/drive/MyDrive/SRN_dataset/srn_cars/cars_test')


# Function to check if each 'rgb' and 'pose' sub-subfolder pair has same amount of files
def check_cars_train_folder(main_folder, expected_count=50):
    #print(f"Checking folder: {main_folder}")
    for subfolder in os.listdir(main_folder):
        subfolder_path = os.path.join(main_folder, subfolder)
        #print(f"Checking subfolder: {subfolder_path}")

        if os.path.isdir(subfolder_path):
            depth_path = os.path.join(subfolder_path, 'depth')
            rgb_path = os.path.join(subfolder_path, 'rgb')
            pose_path = os.path.join(subfolder_path, 'pose')

            if os.path.exists(rgb_path) and os.path.exists(pose_path) and os.path.exists(depth_path):
                depth_files = [f for f in os.listdir(depth_path) if f.endswith('.png')]
                rgb_files = [f for f in os.listdir(rgb_path) if f.endswith('.png')]
                txt_files = [f for f in os.listdir(pose_path) if f.endswith('.txt')]

                if len(rgb_files) != expected_count or len(txt_files) != expected_count or len(depth_files) != expected_count:
                    print(f"Mismatch in folder: {subfolder_path}")
                    print(f"RGB folder has {len(rgb_files)} PNG files, Pose folder has {len(txt_files)} TXT files, Depth folder has {len(depth_files)} png files, .")

for main in cars_train_path:
  # Run the check on the cars_train folder
  check_cars_train_folder(main, expected_count=50)

print("finished checking all folders have valid number of pose, rgb files")


